# 02 · Feature engineering cookbook

Quick, runnable recipes for the transformations interviewers ask about.
| Data type | Recipes |
|---|---|
| Numeric | scaling (standard, min-max, robust), log transform, binning, interactions/polynomials |
| Categorical | one-hot, ordinal, target encoding (CV-safe), rare-category grouping |
| Date/time | year, month, weekday, hour, cyclical sin/cos, "days since" |
| Text | bag-of-words, TF-IDF, embeddings (module 07) |
| Missing | median/mode impute, indicator flags, KNN/iterative impute |

In [1]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, OrdinalEncoder, TargetEncoder, KBinsDiscretizer
from sklearn.impute import KNNImputer
from sklearn.feature_extraction.text import TfidfVectorizer
rng = np.random.default_rng(0)

## Scaling: which scaler when?

In [2]:
x = np.concatenate([rng.normal(50, 10, 98), [300, 400]]).reshape(-1, 1)   # two outliers
out = pd.DataFrame({
    "raw": x.ravel(),
    "standard": StandardScaler().fit_transform(x).ravel(),
    "minmax": MinMaxScaler().fit_transform(x).ravel(),
    "robust (median/IQR)": RobustScaler().fit_transform(x).ravel(),
    "log1p": np.log1p(x).ravel(),
})
out.describe().round(2)

,raw,standard,minmax,robust (median/IQR),log1p
count,100.00,100.00,100.00,100.00,100.00
mean,57.09,-0.00,0.08,0.41,3.97
std,43.68,1.01,0.12,3.05,0.33
min,26.75,-0.70,0.00,-1.70,3.32
25%,43.81,-0.31,0.05,-0.51,3.80
50%,51.15,-0.14,0.07,0.00,3.95
75%,58.14,0.02,0.08,0.49,4.08
max,400.00,7.89,1.00,24.34,5.99


- Tree models don't need scaling. Distance/gradient-based models (KNN, SVM, linear, neural nets) do.
- Outliers squash min-max scaling → use robust scaling or a log transform for skewed positive data.

## Categorical encodings

In [3]:
cities = pd.DataFrame({"city": ["Dhaka", "Chattogram", "Dhaka", "Sylhet", "Khulna", "Dhaka"],
                       "size": ["L", "M", "L", "S", "M", "L"]})
print(OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(cities[["city"]]).get_feature_names_out())
print(OrdinalEncoder(categories=[["S", "M", "L"]]).fit_transform(cities[["size"]]).ravel())

['city_Chattogram' 'city_Dhaka' 'city_Khulna' 'city_Sylhet']
[2. 1. 2. 0. 1. 2.]


**Target encoding** replaces a category by the mean target for that category. It leaks unless it is
computed out-of-fold. sklearn's `TargetEncoder` does cross-fitting inside `fit_transform` automatically.

In [4]:
n = 1000
cat = rng.choice([f"c{i}" for i in range(50)], n)
effect = {f"c{i}": rng.normal() for i in range(50)}
y = (np.array([effect[c] for c in cat]) + rng.normal(0, 1, n) > 0).astype(int)
te = TargetEncoder(target_type="binary", random_state=0)
encoded = te.fit_transform(cat.reshape(-1, 1), y)     # cross-fitted: safe for training
print(encoded[:5].ravel().round(3))

[0.614 1.    0.    0.654 0.301]


## Dates: cyclical encoding so 23:00 is close to 00:00

In [5]:
ts = pd.Series(pd.date_range("2026-01-01", periods=6, freq="7h"))
feats = pd.DataFrame({"ts": ts, "month": ts.dt.month, "weekday": ts.dt.weekday, "hour": ts.dt.hour,
                      "hour_sin": np.sin(2 * np.pi * ts.dt.hour / 24), "hour_cos": np.cos(2 * np.pi * ts.dt.hour / 24)})
feats.round(3)

,ts,month,weekday,hour,hour_sin,hour_cos
0,2026-01-01 00:00:00,1,3,0,0.000,1.000
1,2026-01-01 07:00:00,1,3,7,0.966,-0.259
2,2026-01-01 14:00:00,1,3,14,-0.500,-0.866
3,2026-01-01 21:00:00,1,3,21,-0.707,0.707
4,2026-01-02 04:00:00,1,4,4,0.866,0.500
5,2026-01-02 11:00:00,1,4,11,0.259,-0.966


## Binning and interactions

In [6]:
ages = rng.integers(18, 90, 10).reshape(-1, 1)
print(np.c_[ages, KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile").fit_transform(ages)])
d = pd.DataFrame({"weight_kg": [60, 80, 95], "height_m": [1.6, 1.75, 1.8]})
d["bmi"] = d.weight_kg / d.height_m ** 2          # domain interaction beats blind polynomials
d

[[54.  1.]
 [83.  3.]
 [53.  1.]
 [59.  2.]
 [83.  3.]
 [21.  0.]
 [45.  0.]
 [73.  3.]
 [64.  2.]
 [41.  0.]]


,weight_kg,height_m,bmi
0,60,1.60,23.437500
1,80,1.75,26.122449
2,95,1.80,29.320988


## Missing values: KNN imputation

In [7]:
m = np.array([[1, 2, np.nan], [3, 4, 3], [np.nan, 6, 5], [8, 8, 7]], dtype=float)
KNNImputer(n_neighbors=2).fit_transform(m)

array([[1. , 2. , 4. ],
       [3. , 4. , 3. ],
       [5.5, 6. , 5. ],
       [8. , 8. , 7. ]])

## Text → TF-IDF

In [8]:
docs = ["patient has fever and cough", "no fever reported", "severe cough and chest pain"]
tf = TfidfVectorizer()
mat = tf.fit_transform(docs)
pd.DataFrame(mat.toarray(), columns=tf.get_feature_names_out()).round(2)

,and,chest,cough,fever,has,no,pain,patient,reported,severe
0,0.39,0.00,0.39,0.39,0.52,0.00,0.00,0.52,0.00,0.00
1,0.00,0.00,0.00,0.47,0.00,0.62,0.00,0.00,0.62,0.00
2,0.37,0.49,0.37,0.00,0.00,0.00,0.49,0.00,0.00,0.49


## ✍️ Your turn
1. Add cyclical month features to a dataset with dates, and compare a model with and without them.
2. Show target-encoding leakage: compute category means on **all** data (no cross-fitting) and compare CV scores.

## 🎯 Interview check
- One-hot vs target encoding for a column with 10,000 categories?
- Why encode hours with sin/cos?
- Which models need feature scaling and why?

In [9]:
# Your code here